# 🎨 TRELLIS: Image to 3D GLB Converter

Convert images to high-quality 3D models using Microsoft's TRELLIS!

## Before You Start

1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `T4 GPU` or `A100 GPU`
2. **Check GPU**: Run the cell below to verify GPU is available
3. **Upload Images**: You'll be able to upload images later in the notebook

---

## ✅ Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ WARNING: No GPU detected! Please enable GPU in Runtime settings.")

## 📦 Step 2: Install Dependencies (FIXED)

This will take 5-10 minutes on first run. Subsequent runs will be faster.

In [ ]:
import os
import sys
from pathlib import Path

print("📦 Installing dependencies...\n")

# Install system dependencies
print("[1/5] Installing system packages...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq git git-lfs > /dev/null 2>&1

# Install compatible PyTorch versions first
print("[2/5] Installing PyTorch (this may take a few minutes)...")
!pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

# Install other dependencies
print("[3/5] Installing core dependencies...")
!pip install -q transformers diffusers accelerate
!pip install -q pillow numpy scipy
!pip install -q trimesh[easy] pygltflib
!pip install -q huggingface_hub einops omegaconf
!pip install -q xformers

# Clone TRELLIS repository
print("[4/5] Setting up TRELLIS...")
TRELLIS_DIR = Path("/content/TRELLIS")
if not TRELLIS_DIR.exists():
    !git clone -q --recursive https://github.com/microsoft/TRELLIS.git /content/TRELLIS
    print("   ✓ TRELLIS cloned")
else:
    print("   ✓ TRELLIS already exists")

# Add TRELLIS to Python path
if str(TRELLIS_DIR) not in sys.path:
    sys.path.insert(0, str(TRELLIS_DIR))

# Install additional tool dependencies
print("[5/5] Installing utility packages...")
!pip install -q click pyyaml tqdm rich matplotlib

print("\n✅ Installation complete!")
print("\nVerifying installation...")
import torch
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  TRELLIS path: {TRELLIS_DIR}")

## 🚀 Step 3: Load TRELLIS Model

This downloads the model (~5GB) and loads it into GPU memory. Only needs to run once per session.

In [ ]:
print("Loading TRELLIS pipeline...\n")

import torch
from PIL import Image
import numpy as np

# Import TRELLIS
try:
    from trellis.pipelines import TrellisImageTo3DPipeline
    print("✓ TRELLIS imported successfully")
except ImportError as e:
    print(f"❌ Failed to import TRELLIS: {e}")
    print("\nTrying alternative import...")
    import sys
    sys.path.insert(0, '/content/TRELLIS')
    from trellis.pipelines import TrellisImageTo3DPipeline

# Configuration
MODEL_NAME = "JeffreyXiang/TRELLIS-image-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nModel: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print("\nDownloading model (first run only, ~5GB)...")
print("This may take 3-5 minutes...\n")

# Load pipeline
pipeline = TrellisImageTo3DPipeline.from_pretrained(MODEL_NAME)
pipeline = pipeline.to(DEVICE)

print("\n✅ Model loaded successfully!")
if torch.cuda.is_available():
    print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 🛠️ Step 4: Helper Functions

In [ ]:
import trimesh
from pathlib import Path
from IPython.display import display, Image as IPImage, FileLink
import matplotlib.pyplot as plt

def process_image_to_3d(image_path, output_path="output.glb", seed=1, show_preview=True):
    """
    Convert an image to a 3D GLB file.
    
    Args:
        image_path: Path to input image
        output_path: Path for output GLB file
        seed: Random seed for reproducibility
        show_preview: Show input image preview
    
    Returns:
        Path to generated GLB file
    """
    # Load image
    print(f"📷 Loading image: {Path(image_path).name}")
    image = Image.open(image_path)
    
    if image.mode != "RGB":
        print(f"   Converting from {image.mode} to RGB")
        image = image.convert("RGB")
    
    print(f"   Size: {image.size[0]}x{image.size[1]} pixels")
    
    # Show preview
    if show_preview:
        plt.figure(figsize=(6, 6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"Input: {Path(image_path).name}", fontsize=12)
        plt.tight_layout()
        plt.show()
    
    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Run TRELLIS inference
    print(f"\n🔮 Running TRELLIS inference (seed={seed})...")
    print("   This takes ~30-60 seconds on T4 GPU...")
    
    try:
        outputs = pipeline.run(image, seed=seed)
        print("   ✓ Inference complete!")
    except Exception as e:
        print(f"   ❌ Inference failed: {e}")
        return None
    
    # Extract and export mesh
    print(f"\n💾 Exporting to GLB...")
    
    try:
        # TRELLIS outputs are typically: gaussian, radiance_field, mesh
        # Try different access methods
        mesh_data = None
        
        if hasattr(outputs, 'mesh'):
            mesh_data = outputs.mesh
        elif isinstance(outputs, dict) and 'mesh' in outputs:
            mesh_data = outputs['mesh']
        elif hasattr(outputs, 'get_mesh'):
            mesh_data = outputs.get_mesh()
        
        if mesh_data is None:
            print(f"   ⚠️ Could not find mesh in outputs")
            print(f"   Output type: {type(outputs)}")
            if isinstance(outputs, dict):
                print(f"   Available keys: {list(outputs.keys())}")
            else:
                print(f"   Available attributes: {[a for a in dir(outputs) if not a.startswith('_')]}")
            return None
        
        # Export mesh
        if hasattr(mesh_data, 'export'):
            # Use TRELLIS built-in export
            mesh_data.export(output_path)
        else:
            # Manual export with trimesh
            vertices = None
            faces = None
            
            # Try different attribute names
            for v_name in ['vertices', 'verts', 'v']:
                if hasattr(mesh_data, v_name):
                    vertices = getattr(mesh_data, v_name)
                    break
                elif isinstance(mesh_data, dict) and v_name in mesh_data:
                    vertices = mesh_data[v_name]
                    break
            
            for f_name in ['faces', 'tris', 'f']:
                if hasattr(mesh_data, f_name):
                    faces = getattr(mesh_data, f_name)
                    break
                elif isinstance(mesh_data, dict) and f_name in mesh_data:
                    faces = mesh_data[f_name]
                    break
            
            if vertices is None or faces is None:
                print(f"   ❌ Could not extract vertices/faces")
                return None
            
            # Convert to numpy if needed
            if torch.is_tensor(vertices):
                vertices = vertices.detach().cpu().numpy()
            if torch.is_tensor(faces):
                faces = faces.detach().cpu().numpy()
            
            # Create and export trimesh
            mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
            mesh.export(output_path, file_type='glb')
        
        # Verify output
        if Path(output_path).exists():
            file_size = Path(output_path).stat().st_size / 1e6
            print(f"\n✅ Success!")
            print(f"   Output: {output_path}")
            print(f"   Size: {file_size:.2f} MB")
            return output_path
        else:
            print(f"   ❌ Output file not created")
            return None
        
    except Exception as e:
        print(f"   ❌ Export failed: {e}")
        import traceback
        traceback.print_exc()
        return None

def batch_process(image_dir, output_dir="/content/outputs", seed=1):
    """
    Process all images in a directory.
    
    Args:
        image_dir: Directory containing images
        output_dir: Directory for output GLB files
        seed: Random seed
    """
    import glob
    
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    # Find images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.JPG', '*.JPEG', '*.PNG']
    images = []
    for ext in extensions:
        images.extend(glob.glob(f"{image_dir}/{ext}"))
    
    if not images:
        print(f"❌ No images found in {image_dir}")
        return []
    
    print(f"📁 Found {len(images)} images\n")
    
    results = []
    for i, img_path in enumerate(images, 1):
        print(f"\n{'='*70}")
        print(f"[{i}/{len(images)}] Processing: {Path(img_path).name}")
        print(f"{'='*70}")
        
        output_name = Path(img_path).stem + ".glb"
        output_path = Path(output_dir) / output_name
        
        try:
            result = process_image_to_3d(img_path, str(output_path), seed=seed, show_preview=False)
            if result:
                results.append(result)
        except Exception as e:
            print(f"❌ Failed: {e}")
            continue
    
    print(f"\n\n{'='*70}")
    print(f"✅ Batch Complete: {len(results)}/{len(images)} successful")
    print(f"{'='*70}")
    print(f"📁 Output directory: {output_dir}")
    
    return results

print("✅ Helper functions loaded!")

## 📤 Step 5: Upload Images

Upload one or more images to convert to 3D.

In [ ]:
from google.colab import files
import shutil

# Create input directory
input_dir = Path("/content/inputs")
input_dir.mkdir(exist_ok=True)

print("📤 Click 'Choose Files' below to upload your images...\n")
uploaded = files.upload()

if not uploaded:
    print("\n⚠️ No files uploaded")
else:
    # Move uploaded files to input directory
    print("\n✅ Uploaded:")
    for filename in uploaded.keys():
        src = Path(filename)
        dst = input_dir / filename
        if src.exists():
            shutil.move(str(src), str(dst))
        print(f"   • {filename} ({len(uploaded[filename]) / 1e6:.2f} MB)")
    
    print(f"\n📁 Files saved to: {input_dir}")

## 🎨 Step 6: Convert Single Image

Convert one image to 3D GLB format.

In [ ]:
# List available images
print("📁 Available images:\n")
available_images = []
for img in sorted(input_dir.glob("*")):
    if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        print(f"   • {img.name}")
        available_images.append(img)

if not available_images:
    print("   ⚠️ No images found. Please upload images in Step 5 first.")
else:
    print(f"\n{'='*70}")
    
    # Configuration
    IMAGE_PATH = str(available_images[0])  # Convert first image by default
    OUTPUT_PATH = "/content/output.glb"
    SEED = 1  # Change for different variations
    
    print(f"Converting: {Path(IMAGE_PATH).name}")
    print(f"Seed: {SEED}")
    print(f"{'='*70}\n")
    
    # Convert
    result = process_image_to_3d(IMAGE_PATH, OUTPUT_PATH, seed=SEED, show_preview=True)
    
    if result:
        print(f"\n{'='*70}")
        print("📥 Download your 3D model below:")
        print(f"{'='*70}")
        display(FileLink(result))
        
        print("\n💡 Tip: View your GLB file at https://gltf-viewer.donmccurdy.com")

## 🔄 Step 7: Batch Convert (Optional)

Convert ALL uploaded images at once.

In [ ]:
OUTPUT_DIR = "/content/outputs"
SEED = 1

# Process all images
results = batch_process(str(input_dir), OUTPUT_DIR, seed=SEED)

if results:
    # List outputs
    print(f"\n{'='*70}")
    print("📥 Download individual models:")
    print(f"{'='*70}\n")
    
    for glb_file in sorted(Path(OUTPUT_DIR).glob("*.glb")):
        file_size = glb_file.stat().st_size / 1e6
        print(f"   {glb_file.name} ({file_size:.2f} MB)")
        display(FileLink(str(glb_file)))

## 📦 Step 8: Download All Outputs as ZIP (Optional)

Download all generated GLB files in one zip archive.

In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = "/content/outputs"

# Check if outputs exist
glb_files = list(Path(OUTPUT_DIR).glob("*.glb"))

if not glb_files:
    print("❌ No GLB files found. Run Step 7 first to generate models.")
else:
    print(f"📦 Creating zip archive with {len(glb_files)} models...")
    
    # Create zip archive
    archive_path = shutil.make_archive("/content/3d_models", 'zip', OUTPUT_DIR)
    archive_size = Path(archive_path).stat().st_size / 1e6
    
    print(f"   ✓ Archive created: 3d_models.zip ({archive_size:.2f} MB)")
    print("\n📥 Downloading...")
    
    files.download(archive_path)
    
    print("\n✅ Download complete!")

## 👁️ View Your 3D Models

You can view your GLB files using:

### Online Viewers (Easiest)
1. **[gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com)** - Simple drag & drop viewer
2. **[Babylon.js Sandbox](https://sandbox.babylonjs.com)** - Advanced viewer with lighting
3. **[Three.js Editor](https://threejs.org/editor/)** - Edit and inspect models

### Desktop Applications
4. **Blender** (Free) - File → Import → glTF 2.0
5. **Windows 3D Viewer** - Built-in Windows app
6. **Paint 3D** - Windows 10/11

### Mobile Apps
7. **Sketchfab** - iOS/Android
8. **Canvas** - 3D viewer apps

### Game Engines
9. **Unity** - Import as 3D asset
10. **Unreal Engine** - Direct GLB import

---

## 💡 Tips for Best Results

### Image Selection
✅ **Good:**
- Clear, well-lit objects
- Single prominent subject
- Minimal background clutter
- High resolution (1024x1024+)
- Objects with visible depth/shape

❌ **Avoid:**
- Flat 2D images or logos
- Very cluttered scenes
- Poor/dark lighting
- Extremely low resolution
- Blurry or out-of-focus

### Example Good Subjects
- 🪑 Furniture (chairs, tables, lamps)
- 🚗 Vehicles (cars, bikes, boats)
- 👟 Products (shoes, bottles, gadgets)
- 🧸 Characters/figurines
- 🍎 Food items
- 🌿 Plants/flowers

### Performance
- **T4 GPU**: ~30-60 seconds per image
- **A100 GPU**: ~15-30 seconds per image
- **Seeds**: Try different seeds (1-100) for variations
- **Resolution**: Higher input = better detail (but same time)

---

## 🐛 Troubleshooting

### Out of Memory
```python
# Restart runtime
Runtime → Restart runtime
# Then process one image at a time
```

### Model Load Fails
- Check internet connection
- Model download is ~5GB, may take time
- Try restarting runtime and re-running Step 3

### Export Fails
- Check the debug output for clues
- TRELLIS API may have changed
- Try restarting from Step 1

### Session Disconnected
- Free tier has time limits (~2-3 hours)
- Download your GLB files before closing
- Models stay cached on restart (faster)

### "No GPU Available"
- Runtime → Change runtime type → T4 GPU
- Free tier may have wait times
- Consider Colab Pro for priority access

---

## 📚 Resources

- **TRELLIS GitHub**: [github.com/microsoft/TRELLIS](https://github.com/microsoft/TRELLIS)
- **TRELLIS Paper**: Research publication on arXiv
- **GLB Format**: [khronos.org/gltf](https://www.khronos.org/gltf/)
- **Trimesh Docs**: [trimsh.org](https://trimsh.org)

---

## 🎉 Enjoy Creating 3D Models!

Share your creations:
- Upload to [Sketchfab](https://sketchfab.com)
- Post on social media
- Use in your projects

**Happy 3D modeling!** ✨